# 🛡️ Glu-Stock: 01_RESEARCH_SCAN
**Phase**: Universe Selection & Fundamental Filtering

This notebook scans the IDX market, filters for liquidity and high-growth fundamentals, and pushes candidates to the Firebase `research` queue.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib python-dotenv


In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Web Fetchers)\nimport json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf\nfrom firebase_admin import credentials, firestore\nfrom datetime import datetime\n\ntry:\n    from kaggle_secrets import UserSecretsClient\n    IS_KAGGLE = True\nexcept ImportError:\n    IS_KAGGLE = False\n\nclass KaggleInfra:\n    @staticmethod\n    def load_secrets():\n        if IS_KAGGLE:\n            user_secrets = UserSecretsClient()\n            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")\n            except: tg = None\n            return {\n                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),\n                "telegram": tg\n            }\n        else:\n            from dotenv import load_dotenv\n            load_dotenv()\n            return {\n                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),\n                "telegram": os.getenv("TELEGRAM_TOKEN")\n            }\n\nclass FirebaseHandler:\n    def __init__(self, secrets):\n        if not firebase_admin._apps:\n            cred = credentials.Certificate(secrets['key'])\n            firebase_admin.initialize_app(cred)\n        self.db = firestore.client()\n        \n    def get_and_clear_queue(self, queue_name: str):\n        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()\n        tasks = []\n        for doc in docs:\n            dt = doc.to_dict()\n            tasks.append(dt.get('payload', dt))\n            doc.reference.delete()\n        return tasks\n        \n    def push_task(self, queue_name: str, data):\n        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})\n        \n    def insert_trade(self, trade_data):\n        self.db.collection("glu_stock_trades").add(trade_data)\n        \n    def get_history(self, limit=5):\n        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()\n        history = [doc.to_dict() for doc in docs]\n        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}\n        \n    def get_active_trades(self):\n        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()\n        return [doc.to_dict() for doc in docs]\n        \n    def log_event(self, phase, details):\n        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})\n\ndef get_dynamic_lq45():\n    print("🌐 Fetching latest LQ45 constituents...")\n    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]\n    try:\n        import urllib.request\n        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})\n        with urllib.request.urlopen(req, timeout=5) as url:\n            data = json.loads(url.read().decode())\n            return [f"{t}.JK" for t in data]\n    except Exception as e:\n        print(f"⚠️ Github LQ45 fetch failed: {e}. Using highly-curated fallback LQ45 list.")\n    return fallback\n\ndef get_full_idx_universe():\n    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")\n    fallback = ['AALI.JK', 'ABMM.JK', 'ACES.JK', 'ADHI.JK', 'AISA.JK', 'AKRA.JK', 'AMRT.JK', 'ANTM.JK', 'APLN.JK', 'ARNA.JK', 'ARTO.JK', 'ASGR.JK', 'ASII.JK', 'ASRI.JK', 'ASSA.JK', 'AUTO.JK', 'BACA.JK', 'BALI.JK', 'BAYU.JK', 'BBCA.JK', 'BBHI.JK', 'BBNI.JK', 'BBRI.JK', 'BBTN.JK', 'BBYB.JK', 'BCAP.JK', 'BDMN.JK', 'BEST.JK', 'BFIN.JK', 'BGTG.JK', 'BINA.JK', 'BIRD.JK', 'BISI.JK', 'BJBR.JK', 'BJTM.JK', 'BKSL.JK', 'BMRI.JK', 'BMTR.JK', 'BNGA.JK', 'BNII.JK', 'BNLI.JK', 'BRMS.JK', 'BRPT.JK', 'BSDE.JK', 'BSIM.JK', 'BTPN.JK', 'BUDI.JK', 'BUKK.JK', 'BUMI.JK', 'BVIC.JK', 'BWPT.JK', 'BYAN.JK', 'CASS.JK', 'CFIN.JK', 'CITA.JK', 'CMNP.JK', 'CPIN.JK', 'CTRA.JK', 'DEWA.JK', 'DILD.JK', 'DLTA.JK', 'DMAS.JK', 'DNET.JK', 'DOID.JK', 'DSNG.JK', 'DSSA.JK', 'ELSA.JK', 'EMTK.JK', 'ENRG.JK', 'ERAA.JK', 'ESSA.JK', 'EXCL.JK', 'GEMS.JK', 'GGRM.JK', 'GJTL.JK', 'GWSA.JK', 'HEXA.JK', 'HMSP.JK', 'HRUM.JK', 'ICBP.JK', 'IMAS.JK', 'IMPC.JK', 'INCO.JK', 'INDF.JK', 'INDY.JK', 'INKP.JK', 'INPC.JK', 'INTP.JK', 'ISAT.JK', 'ISSP.JK', 'ITMG.JK', 'JKON.JK', 'JPFA.JK', 'JRPT.JK', 'JSMR.JK', 'JTPE.JK', 'KBLI.JK', 'KIJA.JK', 'KKGI.JK', 'KLBF.JK', 'KPIG.JK', 'LPKR.JK', 'LPPF.JK', 'LSIP.JK', 'LTLS.JK', 'MAIN.JK', 'MAPI.JK', 'MAYA.JK', 'MBSS.JK', 'MCOR.JK', 'MDKA.JK', 'MEDC.JK', 'MEGA.JK', 'MIDI.JK', 'MIKA.JK', 'MLBI.JK', 'MLIA.JK', 'MLPL.JK', 'MMLP.JK', 'MNCN.JK', 'MPMX.JK', 'MREI.JK', 'MTDL.JK', 'MTLA.JK', 'MYOR.JK', 'NISP.JK', 'PANR.JK', 'PANS.JK', 'PGAS.JK', 'PNBN.JK', 'PNIN.JK', 'PNLF.JK', 'PTBA.JK', 'PTPP.JK', 'PTRO.JK', 'PWON.JK', 'RAJA.JK', 'RALS.JK', 'SAME.JK', 'SCMA.JK', 'SGRO.JK', 'SIDO.JK', 'SILO.JK', 'SIMP.JK', 'SMAR.JK', 'SMBR.JK', 'SMDR.JK', 'SMGR.JK', 'SMMA.JK', 'SMRA.JK', 'SMSM.JK', 'SRTG.JK', 'SSIA.JK', 'SSMS.JK', 'TBIG.JK', 'TBLA.JK', 'TINS.JK', 'TKIM.JK', 'TLKM.JK', 'TMAS.JK', 'TOBA.JK', 'TOTL.JK', 'TOWR.JK', 'TPMA.JK', 'TRIM.JK', 'TSPC.JK', 'ULTJ.JK', 'UNIC.JK', 'UNTR.JK', 'UNVR.JK', 'VICO.JK', 'WIIM.JK', 'WINS.JK', 'WTON.JK', 'SHIP.JK', 'POWR.JK', 'PRDA.JK', 'BRIS.JK', 'CARS.JK', 'CLEO.JK', 'WOOD.JK', 'HRTA.JK', 'MARK.JK', 'MCAS.JK', 'PSSI.JK', 'MORA.JK', 'PBID.JK', 'IPCM.JK', 'BTPS.JK', 'SPTO.JK', 'HEAL.JK', 'TUGU.JK', 'MSIN.JK', 'MAPA.JK', 'IPCC.JK', 'FILM.JK', 'PANI.JK', 'GOOD.JK', 'SKRN.JK', 'BOLA.JK', 'KOTA.JK', 'KEEN.JK', 'TEBE.JK', 'KEJU.JK', 'PSGO.JK', 'UCID.JK', 'CSRA.JK', 'SAMF.JK', 'SGER.JK', 'PNGO.JK', 'BBSI.JK', 'VICI.JK', 'WMUU.JK', 'UNIQ.JK', 'TAPG.JK', 'BMHS.JK', 'MCOL.JK', 'GTSI.JK', 'MTEL.JK', 'CMRY.JK', 'RMKE.JK', 'AVIA.JK', 'DRMA.JK', 'ADMR.JK', 'STAA.JK', 'MTMH.JK', 'TRGU.JK', 'HATM.JK', 'JARR.JK', 'ELPI.JK', 'MKTR.JK', 'OMED.JK', 'SUNI.JK', 'PGEO.JK', 'BDKR.JK', 'CUAN.JK', 'SMIL.JK', 'AMMN.JK', 'MAHA.JK', 'ERAL.JK', 'BREN.JK', 'MSTI.JK', 'ALII.JK', 'GOLF.JK', 'DAAZ.JK', 'AADI.JK', 'MDIY.JK', 'DGWG.JK', 'CBDK.JK', 'MINE.JK', 'PSAT.JK', 'BLOG.JK', 'YUPI.JK', 'MDLA.JK', 'NCKL.JK', 'MBMA.JK', 'RAAM.JK', 'ADRO.JK', 'AGRO.JK']\n    \n    # 1. Try Official IDX API\n    try:\n        import urllib.request\n        hdrs = {\n            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',\n            'Accept': 'application/json, text/plain, */*',\n            'Referer': 'https://www.idx.co.id/'\n        }\n        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers=hdrs)\n        with urllib.request.urlopen(req, timeout=10) as url:\n            data = json.loads(url.read().decode())\n            if 'data' in data:\n                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]\n                if tickers:\n                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")\n                    return list(set(tickers)) \n    except Exception as e:\n        print(f"⚠️ Official IDX API failed: {e}. Trying Github Proxy...")\n        \n    # 2. Try Github Alternative\n    try:\n        import urllib.request\n        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0'})\n        with urllib.request.urlopen(req, timeout=10) as url:\n            data = json.loads(url.read().decode())\n            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]\n            if tickers:\n                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")\n                return list(set(tickers))\n    except Exception as e:\n        print(f"⚠️ Full fetch failed: {e}. Falling back to MASTER 259 Papan Utama list.")\n        \n    return fallback\n\n

In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Research Agent)
import yfinance as yf
import pandas as pd

class ResearchAgent:
    def get_market_data(self, ticker: str, period: str = "1y") -> pd.DataFrame:
        data = yf.download(ticker, period=period, interval="1d", progress=False)
        return data

    def fundamental_filter(self, ticker: str) -> bool:
        try:
            info = yf.Ticker(ticker).info
            # Filter logic: Growth + Valuation
            growth = info.get('earningsQuarterlyGrowth', 0) or 0
            pe = info.get('trailingPE', 100)
            return growth > 0.05 and pe < 25
        except: return False

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_scan():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    research = ResearchAgent()
    
    universe = get_full_idx_universe()
    candidates = []
    
    print(f"🔭 Scanning {len(universe)} stocks...")
    for ticker in universe:
        if research.fundamental_filter(ticker):
            candidates.append(ticker)
            print(f"✅ {ticker} passed fundamental filter.")
            
    if candidates:
        fb.push_task("research", candidates)
        fb.log_event("RESEARCH", f"Pushed {len(candidates)} stocks to queue.")
        print(f"🚀 Successfully pushed analysis tasks for {candidates}")
    else:
        print("💤 No opportunities found today.")

run_scan()